# Neural sampling along LD1 (supplementary)

Were brain regions sampled evenly along the LD1 axis? If the number of neurons a session contributes from a region tracked LD1, region composition — not LD1 — could drive a brain-wide average. Rebuilt from `neural/qc/neuronal_yield.ipynb` (which used an old LDA file and an old neuron table) on **exactly the neurons and sessions behind fig_neural**: the newest `lda1_neural_pr90_fr10_sess_<date>` tables picked by the date in their name (a copy from a backup keeps an old mtime, which `load_tables` would otherwise go by), `apply_cohort(..., 'common')`, the current LDA file's LD1. Neurons = the firing-rate table's rows (the Fano-factor table is a subset of them).

| panel | what |
|---|---|
| a | every recording session placed at its LD1, one row per broad (Cosmos) region; dot area ∝ neurons it contributes there |
| b | per broad region: Spearman ρ between a session's neuron count in the region (0 where the region was not recorded) and its LD1, across sessions; 95 % CI from sessions resampled; filled = BH q < 0.05 over the broad regions |
| c | total neurons per session against LD1 (Spearman) |
| d | as b, for every fine (Beryl) region recorded in ≥ `MIN_SESSIONS_FINE` sessions; BH over those regions |

**Read with:** the neural tests already include the fine region as a covariate, so a region's mean firing rate differing from another's cannot masquerade as an LD1 effect; what a yield–LD1 correlation would change is how much each region weighs in the brain-wide estimate.

In [ ]:
import sys, pathlib, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests
from datetime import date

FIGDIR = pathlib.Path.cwd().resolve()
while not (FIGDIR / 'fig_layout.py').exists() and FIGDIR != FIGDIR.parent:
    FIGDIR = FIGDIR / 'figure_generation' if (FIGDIR / 'figure_generation').exists() else FIGDIR.parent
PI = FIGDIR.parent
for p in [str(PI), str(PI / 'neural'), str(FIGDIR)]:
    if p not in sys.path:
        sys.path.insert(0, p)
# the LDA file was pickled under numpy 2; under numpy 1 its module paths need aliasing to unpickle
try:
    import numpy._core.numeric  # noqa: F401
except ModuleNotFoundError:
    import numpy.core.numeric, numpy.core.multiarray
    sys.modules['numpy._core.numeric'], sys.modules['numpy._core.multiarray'] = np.core.numeric, np.core.multiarray
import paper_style as ps
import fig_layout as fl
import lda1_neural_metrics as nm
import run_behavior_axes as rba
from iblatlas.regions import BrainRegions
ps.use('paper')
FIG = 'fig_neural_sampling'
DARK, LIGHT = '#12355B', '#7FB0D8'
MIN_SESSIONS_FINE = 10      # d: fine regions recorded in at least this many sessions
N_BOOT = 2000
SEED = 0

## The cohort of fig_neural

In [ ]:
cfg = rba.config()
want = nm.cache_tag(cfg)
builds = sorted({f.name.split('__')[0] for f in nm.CACHE.glob(f'{want}_*__ff.pqt')
                 if re.fullmatch(r'\d{2}-\d{2}-\d{4}', f.name.split('__')[0][len(want) + 1:])},
                key=lambda b: pd.to_datetime(b[-10:], dayfirst=True))
T = nm.apply_cohort(nm.load_tables(tag=builds[-1], verbose=False), cfg, 'common', verbose=False)
NEU = T['fr'][['session', 'mouse_name', 'region', 'lda_1']].dropna(subset=['lda_1'])
br = BrainRegions()
_u = NEU['region'].unique()
NEU['cosmos'] = NEU['region'].map(dict(zip(_u, br.acronym2acronym(_u, mapping='Cosmos'))))
FULL = {a: n.replace(' (left)', '').replace(' (right)', '') for a, n in zip(br.acronym, br.name)}
SESS = NEU.groupby('session').agg(lda_1=('lda_1', 'first'), mouse_name=('mouse_name', 'first'),
                                  n_neurons=('region', 'size'))
print(f"{builds[-1]}: {len(NEU)} neurons, {len(SESS)} sessions, {SESS['mouse_name'].nunique()} mice, "
      f"{NEU['cosmos'].nunique()} broad / {NEU['region'].nunique()} fine regions")

## Yield against LD1

Per region, the session × region neuron count with **zeros for sessions that did not record the region**, so the correlation covers both whether a region was sampled and how densely.

In [ ]:
def yield_table(level, min_sessions=1):
    counts = NEU.groupby(['session', level]).size().unstack(fill_value=0).reindex(SESS.index, fill_value=0)
    x = SESS['lda_1'].to_numpy()
    rng = np.random.default_rng(SEED)
    boots = [rng.integers(0, len(x), len(x)) for _ in range(N_BOOT)]
    rows = []
    for reg in counts.columns:
        y = counts[reg].to_numpy()
        n_sess = int((y > 0).sum())
        if n_sess < min_sessions:
            continue
        rho, p = spearmanr(x, y)
        bs = [spearmanr(x[b], y[b])[0] for b in boots]
        lo, hi = np.nanpercentile(bs, [2.5, 97.5])
        rows.append(dict(region=reg, name=FULL.get(reg, reg), n_sessions=n_sess, n_neurons=int(y.sum()),
                         rho=rho, p=p, lo=lo, hi=hi))
    t = pd.DataFrame(rows)
    t['q'] = multipletests(t['p'], method='fdr_bh')[1]
    return t.sort_values('n_neurons', ascending=False).reset_index(drop=True), counts

BROAD, BROAD_COUNTS = yield_table('cosmos')
BROAD = BROAD[~BROAD['region'].isin(['root', 'void'])].reset_index(drop=True)
FINE, _ = yield_table('region', MIN_SESSIONS_FINE)
TOTAL = spearmanr(SESS['lda_1'], SESS['n_neurons'])
print(BROAD.round(3).to_string(index=False))
print(f"\nfine regions in >= {MIN_SESSIONS_FINE} sessions: {len(FINE)}; q < 0.05: "
      f"{list(FINE.loc[FINE['q'] < 0.05, 'region'])}; p < 0.05 uncorrected: {list(FINE.loc[FINE['p'] < 0.05, 'region'])}")
print(f"total neurons per session vs LD1: rho = {TOTAL[0]:+.3f}, p = {TOTAL[1]:.3f}")

## Panels

In [ ]:
def draw_A(fig, spec):
    ax = fig.add_subplot(spec)
    order = list(BROAD['region'])
    for i, reg in enumerate(order):
        n = BROAD_COUNTS[reg]
        s = n[n > 0]
        ax.scatter(SESS.loc[s.index, 'lda_1'], np.full(len(s), i), s=3 + 0.45 * s.to_numpy(), color=DARK, alpha=0.35,
                   lw=0, zorder=2)
    ax.set_yticks(range(len(order))); ax.set_yticklabels([BROAD.set_index('region').loc[r, 'name'] for r in order])
    ax.set_ylim(len(order) - 0.4, -0.6); ax.tick_params(axis='y', length=0)
    ax.spines['left'].set_visible(False)
    for i in range(len(order)):
        ax.axhline(i, color='0.92', lw=0.5, zorder=0)
    ax.set_xlabel('LD1 (session)')
    h = [Line2D([], [], marker='o', ls='none', color=DARK, alpha=0.35, mec='none', ms=np.sqrt(3 + 0.45 * k), label=f'{k}')
         for k in (10, 100, 300)]
    ax.legend(handles=h, title='neurons', frameon=False, loc='lower left', bbox_to_anchor=(0, 1.0), ncol=3,
              handletextpad=0.2, columnspacing=0.8, borderaxespad=0.1)
    return [ax]

def forest(ax, t, vertical=True):
    for i, r in t.reset_index(drop=True).iterrows():
        sig = r['q'] < 0.05
        if vertical:
            ax.plot([r['lo'], r['hi']], [i, i], color=DARK, lw=0.8)
            ax.plot(r['rho'], i, 'o', ms=3.4, color=DARK, mfc=DARK if sig else 'white', mew=0.8, zorder=3)
        else:
            ax.plot([i, i], [r['lo'], r['hi']], color=DARK, lw=0.8)
            ax.plot(i, r['rho'], 'o', ms=3.0, color=DARK, mfc=DARK if sig else 'white', mew=0.8, zorder=3)
    (ax.axvline if vertical else ax.axhline)(0, color='0.6', lw=0.5, zorder=0)

def draw_B(fig, spec, share=None):
    ax = fig.add_subplot(spec, sharey=share)
    forest(ax, BROAD)
    ax.tick_params(labelleft=False, axis='y', length=0); ax.spines['left'].set_visible(False)
    ax.set_xlim(-0.5, 0.5); ax.set_xticks([-0.4, 0, 0.4])
    ax.set_xlabel('ρ, neurons vs LD1')
    return [ax]

def draw_C(fig, spec):
    ax = fig.add_subplot(spec)
    ax.scatter(SESS['lda_1'], SESS['n_neurons'], s=6, color=DARK, alpha=0.5, lw=0)
    ax.set_xlabel('LD1 (session)'); ax.set_ylabel('Neurons per session')
    ax.text(0.98, 0.97, f"ρ = {TOTAL[0]:+.2f}\np = {TOTAL[1]:.2f}", transform=ax.transAxes, ha='right', va='top')
    ax.set_box_aspect(1)
    return [ax]

def draw_D(fig, spec):
    ax = fig.add_subplot(spec)
    t = FINE.sort_values('n_sessions', ascending=False).reset_index(drop=True)
    forest(ax, t, vertical=False)
    ax.set_xticks(range(len(t))); ax.set_xticklabels(t['region'], rotation=60, ha='right', rotation_mode='anchor')
    ax.set_xlim(-0.8, len(t) - 0.2); ax.set_ylim(-0.6, 0.6); ax.set_yticks([-0.5, 0, 0.5])
    ax.set_ylabel('ρ, neurons\nvs LD1')
    ax.set_xlabel(f'Fine region (≥ {MIN_SESSIONS_FINE} sessions, most-sampled first)')
    h = [Line2D([], [], marker='o', ls='-', lw=0.8, ms=3.2, color=DARK, label='q < 0.05'),
         Line2D([], [], marker='o', ls='-', lw=0.8, ms=3.2, color=DARK, mfc='white', label='n.s.')]
    ax.legend(handles=h, frameon=False, loc='upper right', ncol=2, borderaxespad=0.1)
    return [ax]

## The assembled figure

In [ ]:
# a | b on top (b shares a's rows); c | d below
fig = plt.figure(figsize=(fl.FIG_WIDTH, 4.9))
outer = fig.add_gridspec(2, 1, height_ratios=[1.35, 1], hspace=0.55, left=0.2, right=0.985, top=0.92, bottom=0.12)
top = outer[0].subgridspec(1, 2, width_ratios=[2.6, 0.9], wspace=0.08)
bottom = outer[1].subgridspec(1, 2, width_ratios=[0.75, 2.75], wspace=0.35)
pa = draw_A(fig, top[0])
pb = draw_B(fig, top[1], share=pa[0])
pc = draw_C(fig, bottom[0])
pd_ = draw_D(fig, bottom[1])
fl.place_letters(fig, {'a': pa, 'b': pb, 'c': pc, 'd': pd_}, rows=['ab', 'cd'], left='ac')
plt.show()
OUT = FIGDIR / 'figures'
for path in [OUT / f"{FIG}_{date.today().strftime('%d-%m-%Y')}.svg", OUT / f'{FIG}.pdf', OUT / f'{FIG}.png']:
    fig.savefig(path, dpi=300)
    print('wrote', path.relative_to(PI))